## ETL Project + Flask Dashboard

### Download products.csv

In [1]:
import os
import requests

file_path = "products.csv"
url = "https://raw.githubusercontent.com/MohammedHameds/test1455/refs/heads/main/products.csv"

if os.path.exists(file_path):
    print("File already exists. Redownloading...")
    os.remove(file_path)

response = requests.get(url)
with open(file_path, "wb") as f:
    f.write(response.content)
print("File downloaded successfully.")

File already exists. Redownloading...
File downloaded successfully.


### 🔄 Transformations

**Customers**

* Remove duplicates
* Standardize city
* Validate age
* Handle missing email
* Create `full_name`

In [2]:
import pandas as pd

customers_df = pd.read_csv("customers.csv")

print(f"Number of rows before cleaning: {customers_df.shape[0]}")

customers_df = customers_df.drop_duplicates(subset=["email"])

customers_df["city"] = customers_df["city"].str.title()

customers_df = customers_df[(customers_df["age"] >= 18) & (customers_df["age"] <= 120)]

customers_df['email'] = customers_df['email'].fillna(customers_df['first_name'].str.lower() + '@email.com')

customers_df['full_name'] = customers_df['first_name'] + ' ' + customers_df['last_name']

print(f"Number of rows after cleaning: {customers_df.shape[0]}")

Number of rows before cleaning: 5
Number of rows after cleaning: 3


**Orders**

* Remove duplicate `order_id`
* Validate quantity
* Convert `order_date` to datetime
* Validate `customer_id`

In [3]:
orders_df = pd.read_parquet("orders.parquet")

print(f"Number of rows before cleaning: {orders_df.shape[0]}")

orders_df = orders_df.drop_duplicates(subset=["order_id"])

orders_df = orders_df[orders_df["quantity"] > 0]

orders_df["order_date"] = pd.to_datetime(orders_df["order_date"])

orders_df = orders_df[orders_df["customer_id"].isin(customers_df["customer_id"])]

print(f"Number of rows after cleaning: {orders_df.shape[0]}")

Number of rows before cleaning: 11
Number of rows after cleaning: 7


**Products**

* Remove duplicates
* Standardize product/category
* Validate `unit_price`
* Convert price to numeric

In [4]:
products_df = pd.read_csv("products.csv")

print(f"Number of rows before cleaning: {products_df.shape[0]}")

products_df = products_df.drop_duplicates(subset=["product_id"])

products_df["product_name"] = products_df["product_name"].str.title()
products_df["category"] = products_df["category"].str.title()

products_df["unit_price"] = pd.to_numeric(products_df["unit_price"])

products_df = products_df[products_df["unit_price"] > 0]

print(f"Number of rows after cleaning: {products_df.shape[0]}")

Number of rows before cleaning: 6
Number of rows after cleaning: 6


### 💰 Sales

Merge the three datasets and calculate:

```python
sales["total_amount"] = sales["quantity"] * sales["unit_price"]
```

Final columns:

```text
order_id
customer
product
quantity
unit_price
total_amount
```

In [5]:
sales = (
    orders_df
    .merge(customers_df, on="customer_id", how="left")
    .merge(products_df, on="product_id", how="left")
)

sales["total_amount"] = sales["quantity"] * sales["unit_price"]

rename_columns = {
    "full_name": "customer",
    "product_name": "product",
}
sales = sales.rename(columns=rename_columns)

sales = sales[["order_id", "customer", "product", "quantity", "unit_price", "total_amount"]]

sales_tuples = list(sales.itertuples(index=None, name=None))

print(f"Sales dataframe shape: {sales.shape}")
print(f"Sales tuples length: {len(sales_tuples)}")

Sales dataframe shape: (7, 6)
Sales tuples length: 7


### 🗄️ SQL Server

Load the final `Sales` table into SQL Server.

In [6]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pyodbc

# Load SQL Server credentials from the .env file (see .env.example)
load_dotenv()

driver   = os.getenv("DB_DRIVER")
server   = os.getenv("DB_SERVER")
database = os.getenv("DB_DATABASE")

connection_string = (
    f'DRIVER={driver};'
    f'SERVER={server};'
    f'DATABASE={database};'
    'Trusted_Connection=yes;'
    'Encrypt=yes;'
    'TrustServerCertificate=yes;'
)

engine = create_engine(
    "mssql+pyodbc://", 
    creator=lambda: pyodbc.connect(connection_string)
)

# Load the final Sales table into SQL Server (recreated on every run)
sales.to_sql("Sales", engine, if_exists="replace", index=False)

# Verify the load
with engine.connect() as conn:
    count = conn.exec_driver_sql("SELECT COUNT(*) FROM Sales").scalar()

print(f"{count} rows loaded into the 'Sales' table in SQL Server.")

7 rows loaded into the 'Sales' table in SQL Server.


### 🌐 Flask Dashboard

Create a Flask application that:

1. Connects to SQL Server.
2. Retrieves the `Sales` table.
3. Displays the data in an HTML dashboard.

In [ ]:
import os
from dotenv import load_dotenv
from flask import Flask, render_template
from sqlalchemy import create_engine, text

load_dotenv()

driver   = os.getenv("DB_DRIVER")
server   = os.getenv("DB_SERVER")
database = os.getenv("DB_DATABASE")

connection_string = (
    f'DRIVER={driver};'
    f'SERVER={server};'
    f'DATABASE={database};'
    'Trusted_Connection=yes;'
    'Encrypt=yes;'
    'TrustServerCertificate=yes;'
)

engine = create_engine(
    "mssql+pyodbc://", 
    creator=lambda: pyodbc.connect(connection_string)
)

app = Flask(__name__)

def get_sales_rows():
    """Connect to SQL Server and return the Sales table as a list of dicts."""
    with engine.connect() as conn:
        result = conn.execute(text("SELECT * FROM Sales"))
        # Convert rows to list of dicts
        return [dict(row._mapping) for row in result.fetchall()]

@app.route("/")
def dashboard():
    # Retrieve the Sales table from SQL Server
    rows = get_sales_rows()

    # Summary metrics for the dashboard cards
    total_revenue = sum(r["total_amount"] for r in rows)
    avg_order = total_revenue / len(rows) if rows else 0

    # Render templates/dashboard.html
    return render_template(
        "dashboard.html",
        rows=rows,
        total_revenue=total_revenue,
        avg_order=avg_order,
    )

if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [10/Oct/2026 23:29:09] "GET / HTTP/1.1" 200 -
